# 🧪 Extração de Embeddings com BERT

Neste notebook, vamos transformar texto em números (vetores). Esses vetores capturam o **significado semântico** das palavras.

> **Referência de Estudo:** [Hugging Face Learn](https://huggingface.co/learn)

In [1]:
from transformers import AutoTokenizer, AutoModel
import torch

# 1. Carregando o modelo e o Tokenizer (BERT Base Uncased)
# O Tokenizer quebra o texto em 'pedaços' (tokens) que o modelo entende.
# O AutoModel carrega os pesos pré-treinados do BERT (focado em entendimento).
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

frase = "Generative AI is transforming the world of technology."

# 2. Tokenização
# return_tensors="pt" indica que queremos o resultado como tensores do PyTorch.
inputs = tokenizer(frase, return_tensors="pt")
print(f"Tokens gerados: {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     | Details
-------------------------------------------+------------+--------
cls.seq_relationship.weight                | UNEXPECTED |        
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |        
cls.predictions.transform.dense.weight     | UNEXPECTED |        
cls.seq_relationship.bias                  | UNEXPECTED |        
cls.predictions.transform.dense.bias       | UNEXPECTED |        
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |        
cls.predictions.bias                       | UNEXPECTED |        

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens gerados: ['[CLS]', 'genera', '##tive', 'ai', 'is', 'transforming', 'the', 'world', 'of', 'technology', '.', '[SEP]']


In [2]:
# 3. Gerando os Embeddings (Representações Vetoriais)
# torch.no_grad() desativa o cálculo de gradientes, economizando memória e CPU,
# já que estamos apenas fazendo inferência (leitura), não treinamento.
with torch.no_grad():
    outputs = model(**inputs)

# 'last_hidden_state' contém os vetores finais que representam cada token no contexto da frase.
embeddings = outputs.last_hidden_state

# O shape [1, 12, 768] significa:
# 1: Batch (uma frase processada)
# 12: Quantidade de tokens (incluindo [CLS] e [SEP])
# 768: Dimensões do vetor de cada palavra (sua 'impressão digital' semântica)
print(f"Estrutura dos Embeddings (Batch, Tokens, Dimensões): {embeddings.shape}")

print("\nRepresentação vetorial da primeira palavra real (após o token [CLS]):")
print(embeddings[0][1][:10]) # Mostra apenas os 10 primeiros números do vetor

Estrutura dos Embeddings (Batch, Tokens, Dimensões): torch.Size([1, 12, 768])

Representação vetorial da primeira palavra real (após o token [CLS]):
tensor([-0.1621,  0.2665, -0.1592,  0.1868,  0.2323,  0.0222,  0.0743,  0.0127,
         0.4845, -0.8481])


### 💡 Conclusão

O que vimos aqui é a base das LLMs. Palavras não são tratadas como strings, mas como pontos em um espaço multidimensional (geralmente 768 dimensões no BERT). 

**Dica Educacional:** Palavras com significados próximos (ex: 'Rei' e 'Rainha') terminam com vetores cujas distâncias matemáticas (como a Distância de Cosseno) são muito pequenas.